In [2]:
# read SQuAD
import pandas as pd
import json
import os
from transformers import pipeline
# https://stackoverflow.com/questions/74586892/no-module-named-keras-saving-hdf5-format
# https://huggingface.co/learn/nlp-course/chapter2/6?fw=pt

print(os.getcwd())

C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep


In [2]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

No model was supplied, defaulted to facebook/bart-large-mnli and revision c626438 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445991277694702, 0.11197400093078613, 0.04342690110206604]}

In [3]:
classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9598050713539124},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [9]:
question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

No model was supplied, defaulted to distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

(…)squad/resolve/main/tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

(…)d-distilled-squad/resolve/main/vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

(…)tilled-squad/resolve/main/tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

{'score': 0.6949772238731384, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

In [14]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.logits.shape)


torch.Size([2, 2])


In [ ]:
# Now we load the 3 question answering datasets to tokenize and fit pre-trained model

In [33]:
# Stanford Question Answering Dataset
import json

# Opening JSON 
loc = r'.\Stanford Question Answering Dataset\train-v1.1.json'
with open(loc) as f:
    train = json.load(f)

loc2 = r'.\Stanford Question Answering Dataset\dev-v1.1.json'
with open(loc2) as f:
    dev = json.load(f)

# df = pd.json_normalize(data['data'], 
#                      meta=['title', ['paragraphs', 'context'], 
#                            ['paragraphs', 'qas'], ['publisher', 'location']])
# df.columns = ['Title', 'Author_First_Name', 'Author_Last_Name', 'Publisher_Name', 'Publisher_Location']


In [38]:
# Stanford Question Answering Dataset (SQuAD)
train = pd.read_csv(r"C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep\Stanford Question Answering Dataset (SQuAD)\train.csv")
validation = pd.read_csv(r"C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep\Stanford Question Answering Dataset (SQuAD)\validation.csv")

In [50]:
train[train.id == '56cf6ca44df3c31400b0d777'].transpose()

,1696
id,56cf6ca44df3c31400b0d777
title,Frédéric_Chopin
context,Chopin's successes as a composer and performer...
question,"Who said that Chopin set out ""into the wide wo..."
answers,"{'text': array(['Zdzisław Jachimecki'], dtype=..."


In [ ]:
# Question-Answer Dataset
# qa1 = pd.read_fwf(r"C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep\Question-Answer Dataset\S08_question_answer_pairs.txt", delimiter = "\t")
# qa2 = pd.read_fwf(r"C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep\Question-Answer Dataset\S09_question_answer_pairs.txt")
# qa3 = pd.read_fwf(r"C:\Users\Kathy\develop\UCSD Bootcamp\Capstone Prep\Question-Answer Dataset\S10_question_answer_pairs.txt")


In [ ]:
# run tutorial from here:
# https://huggingface.co/docs/transformers/tasks/question_answering

# extractive question answering
# metrics are exact match and prediction score
# dataset has context, questions, answers
# can use on querying a document, then feeding in questions from customers


In [1]:
from datasets import load_dataset
# same as our train and validation
squad = load_dataset("squad", split="train[:5000]")
squad = squad.train_test_split(test_size=0.2)

In [3]:
squad["train"][0]

{'id': '56cda50b62d2951400fa67ac',
 'title': 'The_Legend_of_Zelda:_Twilight_Princess',
 'context': 'A high-definition remaster of the game, The Legend of Zelda: Twilight Princess HD, is being developed by Tantalus Media for the Wii U. Officially announced during a Nintendo Direct presentation on November 12, 2015, it features enhanced graphics and Amiibo functionality. The game will be released in North America and Europe on March 4, 2016; in Australia on March 5, 2016; and in Japan on March 10, 2016.',
 'question': 'Which company is responsible for the HD version of Twilight Princess?',
 'answers': {'text': ['Tantalus Media'], 'answer_start': [105]}}

In [4]:
from transformers import pipeline, AutoTokenizer
# import torch
# import tensorflow

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [6]:
tokenized_squad = squad.map(preprocess_function, batched=True, 
                            remove_columns=squad["train"].column_names)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
tokenized_squad

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 1000
    })
})

In [8]:
# pytorch? 
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

In [9]:
# pytorch?
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.weight', 'qa_outputs.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
tokenized_squad["train"].shape

(4000, 4)

In [10]:
training_args = TrainingArguments(
    output_dir="my_awesome_qa_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    # push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
# trainer.push_to_hub()


KeyboardInterrupt



In [28]:
# tensorflow?
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator(return_tensors="tf")

In [29]:
# tensorflow?
from transformers import create_optimizer
batch_size = 16
num_epochs = 2
total_train_steps = (len(tokenized_squad["train"]) // batch_size) * num_epochs
optimizer, schedule = create_optimizer(
    init_lr=2e-5,
    num_warmup_steps=0,
    num_train_steps=total_train_steps,
)

ImportError: 
create_optimizer requires the TensorFlow library but it was not found in your environment.
However, we were able to find a PyTorch installation. PyTorch classes do not begin
with "TF", but are otherwise identically named to our TF classes.
If you want to use PyTorch, please use those classes instead!

If you really do want to use TensorFlow, please follow the instructions on the
installation page https://www.tensorflow.org/install that match your environment.


In [ ]:
from transformers import TFAutoModelForQuestionAnswering
model = TFAutoModelForQuestionAnswering("distilbert-base-uncased")

In [ ]:
tf_train_set = model.prepare_tf_dataset(
    tokenized_squad["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

tf_validation_set = model.prepare_tf_dataset(
    tokenized_squad["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

In [ ]:
import tensorflow as tf

model.compile(optimizer=optimizer)

In [ ]:
from transformers.keras_callbacks import PushToHubCallback

callback = PushToHubCallback(
    output_dir="my_awesome_qa_model",
    tokenizer=tokenizer,
)

In [3]:
model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=3, callbacks=[callback])

NameError: name 'model' is not defined

In [ ]:
# NEXT STEP IS EVALUATION

In [ ]:
question = "How many programming languages does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="my_awesome_qa_model")
question_answerer(question=question, context=context)

In [ ]:
# pytorch?
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("my_awesome_qa_model")
inputs = tokenizer(question, context, return_tensors="pt")

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained("my_awesome_qa_model")
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

In [ ]:
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)

In [ ]:
# tensorflow?
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("my_awesome_qa_model")
inputs = tokenizer(question, text, return_tensors="tf")

In [ ]:
from transformers import TFAutoModelForQuestionAnswering

model = TFAutoModelForQuestionAnswering.from_pretrained("my_awesome_qa_model")
outputs = model(**inputs)

In [ ]:
answer_start_index = int(tf.math.argmax(outputs.start_logits, axis=-1)[0])
answer_end_index = int(tf.math.argmax(outputs.end_logits, axis=-1)[0])

In [ ]:
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)

In [8]:
from collections import defaultdict
dct = defaultdict()

for i in range(99):
    s = i % 12 + 1
    dct[str(s)] = dct.get(str(s),0) + 1

In [9]:
dct

defaultdict(None,
            {'1': 9,
             '2': 9,
             '3': 9,
             '4': 8,
             '5': 8,
             '6': 8,
             '7': 8,
             '8': 8,
             '9': 8,
             '10': 8,
             '11': 8,
             '12': 8})

In [11]:
randNum = [0] * 5
mod = 12
cardTokenId = 1432546357465876879867565432301000212

randNum[0] = (cardTokenId % 100) % mod + 1
randNum[1] = ((cardTokenId % 10000) / 100) % mod + 1
randNum[2] = ((cardTokenId % 1000000) / 10000) % mod + 1
randNum[3] = ((cardTokenId % 100000000) / 1000000) % mod + 1
randNum[4] = ((cardTokenId % 10000000000) / 100000000) % mod + 1

print(randNum)

[1, 3.12, 1.0212, 2.0002120000000003, 12.01000212]


In [16]:
((cardTokenId % 10000) / 100)

2.12